# EDA On `game_tracking_2024.csv` Dataset

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

gt24_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "game_tracking_2024.csv"), 
        # Change "NA" to None
        null_values = {
            "event": "NA", 
        }
    )
)

print("Games Data:")
print(f"    - Dimensions: {gt24_df.shape}")
print(f"    - Number of Unique Games: {gt24_df.select("game_id").n_unique()}")
gt24_df

In [ ]:
# =======================================
# Missing Data
# =======================================

# Only event has missing data which is expected

gt24_df.select(pl.all().is_null().sum())

In [ ]:
# =====================================
# game_id
# =====================================

print(f"  - Number of Unique game_id values: {gt24_df.select("game_id").n_unique()}")

In [ ]:
# =====================================
# play_id
# =====================================

print(f"  - Number of Unique play_id values: {gt24_df.select("play_id").n_unique()}")

In [ ]:
# =====================================
# nfl_id
# =====================================

print(f"  - Number of Unique nfl_id values: {gt24_df.select("nfl_id").n_unique()}")

In [ ]:
# ======================================
# Player-Plays
# ======================================

gt24_df_with_pp = (
    gt24_df
    .with_columns(game_play_player = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String))
)

print(f"  - Number of Unique Player-Plays: {gt24_df_with_pp.select("game_play_player").n_unique()}")

In [ ]:
# ==================================================
# x
# ==================================================

# Some x values are NOT within their expected range, but that's very possible.
# If someone is running a route towards then endzone then runs out the back 
# of the endzone.

print(f"  - Min/Max Values of x: [{min(gt24_df["x"])}, {max(gt24_df["x"])}]")

In [ ]:
# ==================================================
# y
# ==================================================

# Some y values are NOT within their expected range, but that's very possible.
# It's not uncommon for players to run out of the sideline and off the field.

print(f"  - Min/Max Values of y: [{min(gt24_df["y"])}, {max(gt24_df["y"])}]")

In [ ]:
# ==================================================
# s (speed)
# ==================================================

max_speed_allowed = 13.69

# NOTE NOTE NOTE: Unlike the combine tracking data, this data has
#                 no unrealistic values of speed

print(f"  - Min/Max Values of s: [{min(gt24_df["s"])}, {max(gt24_df["s"])}]")

# What proportion of speeds are > max allowed speed? 
print(f"  - Percentage of speed values > max allowed speed: {round(100 * gt24_df.filter(pl.col("s") > max_speed_allowed).height / gt24_df.height, 5)}%")

# How many player-plays have speed > max allowed speed?
print(f"  - Percentage of player-plays that have a speed > max allowed speed: {gt24_df_with_pp.filter(pl.col("s") > max_speed_allowed).select("game_play_player").n_unique()}/{gt24_df_with_pp.select("game_play_player").n_unique()} ({round(100 * (gt24_df_with_pp.filter(pl.col("s") > max_speed_allowed).select("game_play_player").n_unique()) / (gt24_df_with_pp.select("game_play_player").n_unique()) , 5)}%)")

In [ ]:
# ==================================================
# a (acceleration)
# ==================================================

max_acceleration_allowed = 10

# According to online resources, max values of acceleration are around 10 y/s^2
# NOTE: Many of these acceleration values are very unrealistic

print(f"  - Min/Max Values of a: [{min(gt24_df["a"])}, {max(gt24_df["a"])}]")

# What proportion of accelerations are > max acceleration allowed? 
print(f"  - Percentage of acceleration values > max acceleration allowed: {round(100 * gt24_df.filter(pl.col("a") > max_acceleration_allowed).height / gt24_df.height, 5)}%")

# How many player-plays have acceleration > max acceleration allowed?
print(f"  - Percentage of player-plays that have a accelration > max acceleration allowed: {gt24_df_with_pp.filter(pl.col("a") > max_acceleration_allowed).select("game_play_player").n_unique()}/{gt24_df_with_pp.select("game_play_player").n_unique()} ({round(100 * (gt24_df_with_pp.filter(pl.col("a") > max_acceleration_allowed).select("game_play_player").n_unique()) / (gt24_df_with_pp.select("game_play_player").n_unique()) , 5)}%)")

In [ ]:
# ==================================================
# dis 
# ==================================================

max_distance_allowed = 2

# According to online resources, max values of dis are around 2 y
# NOTE: Many of these dis values are unrealistic

print(f"  - Min/Max Values of dis: [{min(gt24_df["dis"])}, {max(gt24_df["dis"])}]")

# What proportion of dis are > max allowed distance? 
print(f"  - Percentage of distance values > max allowed distance: {round(100 * gt24_df.filter(pl.col("dis") > max_distance_allowed).height / gt24_df.height, 5)}%")

# How many player-plays have dis > max allowed distance?
print(f"  - Percentage of player-plays that have a distance > max allowed distance: {gt24_df_with_pp.filter(pl.col("dis") > max_distance_allowed).select("game_play_player").n_unique()}/{gt24_df_with_pp.select("game_play_player").n_unique()} ({round(100 * (gt24_df_with_pp.filter(pl.col("dis") > max_distance_allowed).select("game_play_player").n_unique()) / (gt24_df_with_pp.select("game_play_player").n_unique()) , 5)}%)")

In [ ]:
# ========================================
# speed / acceleration / distance
# ========================================

print(f"  - Percentage of player-plays that have a s > 13.69 y/s OR a > 10 y/s^2 OR dis > 2 y: {gt24_df_with_pp.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).select("game_play_player").n_unique()}/{gt24_df_with_pp.select("game_play_player").n_unique()} ({round(100 * (gt24_df_with_pp.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).select("game_play_player").n_unique()) / (gt24_df_with_pp.select("game_play_player").n_unique()) , 5)}%)")
print(f"  - Percentage of     rows     that have a s > 13.69 y/s OR a > 10 y/s^2 OR dis > 2 y: {gt24_df_with_pp.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).height}/{gt24_df_with_pp.height} ({round(100 * gt24_df_with_pp.filter((pl.col("s") > max_speed_allowed) | (pl.col("a") > max_acceleration_allowed) | (pl.col("dis") > max_distance_allowed)).height / gt24_df_with_pp.height, 5)}%)")

In [ ]:
# ==================================================
# dir
# ==================================================

print(f"  - Min/Max Values of dir: [{min(gt24_df["dir"])}, {max(gt24_df["dir"])}]")

In [ ]:
# ==================================================
# o (orientation)
# ==================================================

print(f"  - Min/Max Values of o: [{min(gt24_df["o"])}, {max(gt24_df["o"])}]")

In [ ]:
# ==================================================
# events
# ==================================================

print(f"Number of events (not missing): {gt24_df.filter(pl.col("event").is_not_null()).select("event").n_unique()}")
print(sorted(gt24_df.filter(pl.col("event").is_not_null()).select("event").unique().to_series().to_list()))

gt24_df.group_by("event").agg(count = pl.len()).sort("count", descending = True)